# 模板与插槽

学习目标：能创建模板副本，用插槽复用卡片结构，并区分模板内容、插槽分发和声明式 Shadow DOM。

前置知识：HTML 元素与属性、CSS 选择器与继承，JavaScript 变量、函数、类、DOM 查询和事件监听。

适用范围：WHATWG HTML Living Standard；使用允许 JavaScript 的现代浏览器。声明式 Shadow DOM 的版本条件单独见第 8 节，完整 Web Components 开发留在 Web 应用专题。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/16-templates-and-slots/。

1. [index.html](scripts/16-templates-and-slots/index.html)、[clone.js](scripts/16-templates-and-slots/clone.js)：复制模板、添加笔记和修改模板说明。
2. [slots.html](scripts/16-templates-and-slots/slots.html)、[card.js](scripts/16-templates-and-slots/card.js)：具名、默认及回退插槽，以及最小自定义元素。
3. [slots.js](scripts/16-templates-and-slots/slots.js)：切换插槽名称、编辑后代文字、移动节点并显示分配关系。
4. [declarative.html](scripts/16-templates-and-slots/declarative.html)、[declarative.js](scripts/16-templates-and-slots/declarative.js)：声明式影子根与动态补属性的对照。
5. [styles.css](scripts/16-templates-and-slots/styles.css)：页面排版和样式作用域对照。

## 打开配套页面

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/html
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8016 --bind 127.0.0.1
```

Step 3：打开[模板实例页](http://127.0.0.1:8016/scripts/16-templates-and-slots/index.html)，通过页内导航切换示例。

服务根目录为 content/Web与应用开发/html

Notebook 文件链接相对于本章；页面中的 clone.js 等相对 URL 从页面所在目录解析。普通模板和自定义元素示例需要 JavaScript；声明式影子根本身不依赖页面脚本。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 &lt;template&gt;：保存待复用的结构

普通 &lt;template&gt; 保存一段已解析但不直接显示的内容，需要时由脚本创建副本。本节的模板没有 shadowrootmode；声明式写法见第 7 节。

| 标签 | 中文名称／含义 | 用途 |
| --- | --- | --- |
| &lt;template&gt; | 模板元素 | 保存待复用结构，或声明影子根 |
| &lt;slot&gt; | 插槽元素 | 在影子树内设置内容分发位置 |

下面的模板保存标题、说明和按钮。内部使用 class，避免多次复制后在同一文档中产生重复 id。

```html
<template id="note-template">
  <article class="card">
    <h2 class="note-title">待命名笔记</h2>
    <p class="note-description">第一版阅读目标</p>
    <button type="button" class="finish-note">标记完成</button>
  </article>
</template>
```

配套文件：[scripts/16-templates-and-slots/index.html](scripts/16-templates-and-slots/index.html) · [浏览器预览](http://127.0.0.1:8016/scripts/16-templates-and-slots/index.html)

HTMLTemplateElement 是模板的 DOM 接口；其 content 属性返回 DocumentFragment（文档片段），不是字符串。模板内部节点属于这个片段，不是 &lt;template&gt; 的普通子节点，所以要从 content 查询。

模板内容保持惰性时，其中脚本也不运行。“待命名笔记”只是普通文字，&lt;template&gt; 没有自动数据绑定或变量替换。

## 2 克隆、修改和插入笔记

深克隆（deep clone）复制节点及其后代。clone.js 的添加按钮处理函数先克隆 template.content，再填写副本的标题；true 表示包含后代。

```javascript
const fragment = noteTemplate.content.cloneNode(true);
const card = fragment.querySelector("article");
noteCount += 1;
card.querySelector("h2").textContent = "练习笔记 " + noteCount;
```

配套文件：[scripts/16-templates-and-slots/clone.js](scripts/16-templates-and-slots/clone.js) · [浏览器预览](http://127.0.0.1:8016/scripts/16-templates-and-slots/index.html)

这里 fragment 是副本文档片段，card 是其中的 &lt;article&gt;，noteCount 是已添加的笔记数。本例模板只含原生元素；涉及自定义元素的文档上下文时，可采用第 4 节的 document.importNode。脚本为副本按钮绑定完成操作后，将片段加入页面：

```javascript
notes.appendChild(fragment);
// 预期：副本片段变空，但模板 content 仍有一个 article，可继续克隆。
```

配套文件：[scripts/16-templates-and-slots/clone.js](scripts/16-templates-and-slots/clone.js) · [浏览器预览](http://127.0.0.1:8016/scripts/16-templates-and-slots/index.html)

- 插入 DocumentFragment 会移动它的子节点，片段随即变空；模板原件仍可继续克隆。直接插入 template.content 则会取走模板本身的内容。
- 副本不会与原件持续同步；“修改模板说明”只影响后来添加的笔记。
- cloneNode 不复制通过 addEventListener 添加的监听器，所以完整脚本为每份副本分别绑定完成按钮。

页面保留了两项操作的检查提示：

```html
<!-- 检查：初始无卡片；添加两次后出现两份独立笔记。 -->
<!-- “修改模板说明”不应改变旧卡片，只影响此后克隆的说明。 -->
```

配套文件：[scripts/16-templates-and-slots/index.html](scripts/16-templates-and-slots/index.html) · [浏览器预览](http://127.0.0.1:8016/scripts/16-templates-and-slots/index.html)

## 3 把页面内容交给插槽

复用结构时，可以让页面作者提供内容，由组件内部决定显示位置。Shadow DOM（影子 DOM）为一个宿主元素附着独立子树：

- shadow host：影子宿主，本例为 &lt;study-card&gt;。
- shadow root：影子根，是影子树的根节点。
- shadow tree：影子树，保存组件内部结构。
- light DOM：相对于影子树的普通 DOM，本例指宿主中由页面提供的内容。

&lt;study-card&gt; 是本例注册的自定义元素，初始化代码见下一节。先看页面如何提供标题与正文：

```html
<study-card id="first-card">
  <span id="provided-title" slot="title">周末阅读</span>
  <p id="provided-body">原始正文 <em id="detail">（初稿）</em></p>
  <span id="unmatched" slot="missing">没有同名插槽的内容</span>
</study-card>
```

配套文件：[scripts/16-templates-and-slots/slots.html](scripts/16-templates-and-slots/slots.html) · [浏览器预览](http://127.0.0.1:8016/scripts/16-templates-and-slots/slots.html)

以下三行位于同一页面的卡片模板内，复制到影子根后才进行分发：

```html
<h2><slot name="title">未命名笔记</slot></h2>
<p class="internal-note">内部说明：这段文字来自 shadow tree。</p>
<div><slot><p>没有提供正文</p></slot></div>
```

配套文件：[scripts/16-templates-and-slots/slots.html](scripts/16-templates-and-slots/slots.html) · [浏览器预览](http://127.0.0.1:8016/scripts/16-templates-and-slots/slots.html)

- &lt;slot&gt; 的 name="title"：命名这个插槽，title 是作者约定的名称。
- 提供内容的元素写 slot="title"：分配到同名插槽；slot 属性与 &lt;slot&gt; 标签不是同一对象。
- 不写 name 的 &lt;slot&gt;：默认插槽，接收未命名的直接子元素和直接文本节点。
- 起止标签内的内容：回退内容（fallback content），没有节点分配时显示。

只有宿主的直接子节点参与本层分配；更深后代不能越过父元素单独分配。名称不匹配的 missing 节点留在 light DOM，但不会自动进入默认插槽。多个节点可分给同一插槽；若插槽重名，匹配到树顺序中的第一个。

## 4 连接模板、自定义元素与影子根

card.js 用 StudyCard 类继承 HTMLElement，并通过 customElements.define 注册 study-card。自定义元素名以小写字母开头、包含连字符，还须满足标准的其他命名限制。

在构造器调用 super() 后，以下代码取得模板、创建影子根并插入副本：

```javascript
const template = document.querySelector("#card-template");
const shadow = this.attachShadow({ mode: "open" });
// importNode 在当前 document 的上下文中深克隆模板内容。
// 构造器只初始化影子树，不读取或改写宿主的 light DOM 子节点。
shadow.appendChild(document.importNode(template.content, true));
```

配套文件：[scripts/16-templates-and-slots/card.js](scripts/16-templates-and-slots/card.js) · [浏览器预览](http://127.0.0.1:8016/scripts/16-templates-and-slots/slots.html)

- attachShadow 的 mode="open"：允许通过宿主的 shadowRoot 属性访问影子根。
- document.importNode 的 true：在当前文档上下文中连同后代克隆模板内容。
- 每个宿主都有自己的副本；构造器只初始化影子树，不读取或改写宿主的属性、light DOM 子节点。

slots.html 在 &lt;head&gt; 中按以下顺序加载普通外部脚本，defer 使它们在 HTML 解析后按文档顺序执行：

```html
<script src="card.js" defer></script>
<script src="slots.js" defer></script>
```

配套文件：[scripts/16-templates-and-slots/slots.html](scripts/16-templates-and-slots/slots.html) · [浏览器预览](http://127.0.0.1:8016/scripts/16-templates-and-slots/slots.html)

模板、自定义元素和 Shadow DOM 可配合，也可分别使用：第 2 节没有自定义元素，第 7 节直接以 &lt;div&gt; 为宿主。

## 5 回退、分发与节点移动

回退条件是“没有分配节点”，不是“没有可见文字”。空白文本也能占用默认插槽，下面两种宿主因此不同：

```html
<!-- 两个标签间刻意没有空白，默认插槽才能显示回退正文。 -->
<study-card id="empty-card"></study-card>
<h2>只有一个空格的宿主</h2>
<!-- 一个空格也是 Text 节点，会占用默认插槽；这里不能删除该空格。 -->
<study-card id="space-card"> </study-card>
```

配套文件：[scripts/16-templates-and-slots/slots.html](scripts/16-templates-and-slots/slots.html) · [浏览器预览](http://127.0.0.1:8016/scripts/16-templates-and-slots/slots.html)

插槽分发不会把节点实际移动到 &lt;slot&gt; 下。标题的 parentNode 仍是 first-card，assignedSlot 指向它获得的插槽；插槽的回退内容也仍保留在其 DOM 内。

- assignedNodes()：读取直接分配的节点。无参数时 flatten 默认为 false，不返回回退内容。
- assignedElements()：只返回分配的元素；忽略文本，不能用于判断空白是否占槽。
- assignedNodes({ flatten: true })：会展开插槽，并在没有分配节点时取回退内容。本例状态区使用无参数调用。
- assignedSlot 为 null：在本例 open 根中表示未分配；closed 根也会导致它返回 null，不能无条件据此判断。

页面提供三项操作：

```html
<button type="button" id="toggle-title">切换标题的 slot 名称</button>
<button type="button" id="edit-detail">修改正文内部文字</button>
<button type="button" id="move-body">移动正文到卡片外／移回</button>
```

配套文件：[scripts/16-templates-and-slots/slots.html](scripts/16-templates-and-slots/slots.html) · [浏览器预览](http://127.0.0.1:8016/scripts/16-templates-and-slots/slots.html)

slotchange 用于观察分配节点变化。改变标题的 slot 名称会改变分配；只改已分配正文的后代文字不会触发它。移动按钮通过 appendChild 移动同一个正文节点，才会改变 parentNode。

事件在后续微任务中通知，首次建立分配也可能触发；等页面加载稳定再记下次数，操作后等状态更新再比较，不假定初值为零。完整观察逻辑在 [slots.js](scripts/16-templates-and-slots/slots.js)。

## 6 样式作用域与继承（补充）

普通选择器不会穿过影子边界直接匹配内部节点。页面的 &lt;h2&gt; 规则不匹配卡片内的 &lt;h2&gt;；document.querySelector 也不会自动进入影子树，open 根需从 shadowRoot 查询。

以下样式位于卡片模板中，随副本进入影子根：

```css
:host { display: block; border: 2px solid #6b7280; padding: 16px; margin: 16px 0; }
h2 { color: #7c3c98; }
::slotted([slot="title"]) { border-bottom: 2px dashed #7c3c98; }
```

配套文件：[scripts/16-templates-and-slots/slots.html](scripts/16-templates-and-slots/slots.html) · [浏览器预览](http://127.0.0.1:8016/scripts/16-templates-and-slots/slots.html)

- :host：选择宿主。
- ::slotted([slot="title"])：选择实际分配且 slot 属性值为 title 的元素；不选择纯文本，也不直接匹配这些元素的后代。
- color 等可继承属性可以经宿主传进影子树；已分配节点从插槽继承，但自身的 color 声明优先于继承值。

分配的正文仍在 light DOM，仍可匹配页面 &lt;p&gt; 规则。完整 [styles.css](scripts/16-templates-and-slots/styles.css) 让页面标题为绿色、内部标题为紫色、内部说明继承青色、外部正文为棕色；用开发者工具的 Styles 与 Computed 区分规则匹配和继承。

## 7 在 HTML 中声明影子根

声明式 Shadow DOM（Declarative Shadow DOM，DSD）让浏览器解析 HTML 时建立影子根，无需先执行 attachShadow。给支持作为宿主的父元素放入带 shadowrootmode 的 &lt;template&gt;，本例使用原生 &lt;div&gt;。

```html
<div id="open-host">
  <template shadowrootmode="open">
    <h2>open：解析 HTML 时建立</h2>
    <p>内部说明：不运行页面脚本也应显示这段文字。</p>
    <slot name="label">默认标签</slot>
  </template>
  <span slot="label">外部提供的开放标签</span>
</div>
```

配套文件：[scripts/16-templates-and-slots/declarative.html](scripts/16-templates-and-slots/declarative.html) · [浏览器预览](http://127.0.0.1:8016/scripts/16-templates-and-slots/declarative.html)

shadowrootmode 是枚举属性，必须写有效值：

- open：宿主的 shadowRoot 返回影子根。
- closed：宿主的 shadowRoot 返回 null，内容仍能显示；它不是保护敏感数据的安全隔离。

单写空属性不代表 open。声明成功后，模板内容进入影子根，原位置不留下普通 &lt;template&gt;。一个宿主只能有一个影子根；不要在同一宿主反复声明。

配套页面还有 closed 对照，脚本只读取状态，不负责创建这两个根。对照“查看源代码”和 Elements，辨认声明标记与实际影子树；移除页面脚本引用也不应影响支持浏览器中的声明式内容。

## 8 支持版本与动态属性的边界（补充）

标准 shadowrootmode 的基本声明式写法从 Chrome/Edge 111、Firefox 123、Safari 16.4 起支持。旧教程中的 shadowroot 是旧属性；这些起始版本不代表后来新增的影子根属性也全部可用。

普通 &lt;template&gt; 可用不代表 DSD 可用。declarative.js 同时检查 shadowRootMode 接口、open 根是否存在、模板是否仍保留，并提示观察可见内容；单项接口检测不能证明任意标记都成功建根。

动态创建普通模板后再加属性，不会触发声明式解析。配套脚本先创建 lateTemplate 并把文字放入 content，然后执行：

```javascript
lateTemplate.setAttribute("shadowrootmode", "open");
const lateHost = document.querySelector("#late-host");
lateHost.appendChild(lateTemplate);
```

配套文件：[scripts/16-templates-and-slots/declarative.js](scripts/16-templates-and-slots/declarative.js) · [浏览器预览](http://127.0.0.1:8016/scripts/16-templates-and-slots/declarative.html)

该内容仍在普通模板中。innerHTML 也不是建立声明式影子根的入口；本章把声明直接写进完整 HTML 文件。不支持 DSD 时，模板内容不显示，外部提供的普通文字仍可读。

以后把自定义元素与 DSD 合用时，先处理已经存在的影子根。对匹配模式的声明式根再次调用 attachShadow 会清空并返回已有根，不能无条件调用后假定原内容还在。

## 本章小结

- 普通模板的 content 是文档片段；克隆、修改副本和插入页面各有作用，原件与副本不会自动同步。
- 插槽按名称接收宿主的直接子节点；无分配节点才使用回退，空白也可能占槽。
- 分发改变呈现关系，不改变 light DOM 节点的 DOM 父节点；实际移动要另看节点操作。
- 影子树限制普通查询和选择器的范围，同时保留继承联系。
- DSD 在解析时建立影子根，open、closed 控制常规访问方式，不决定内容是否显示。

自查：页面上的一段文字来自模板副本、light DOM，还是插槽回退内容？

## 练习

在 scripts/16-templates-and-slots/ 内复制需要修改的页面与脚本，并让副本引用对应脚本。

（1）复制 index.html、clone.js 为 practice-template.html、practice-clone.js，为模板增加“阅读时长”。连续添加两张不同分钟数的卡片。检查：没有重复 id，模板仍可继续克隆，完成一张笔记不改变另一张按钮。

（2）复制 slots.html、slots.js 为 practice-slots.html、practice-slots.js，保留对 card.js 的引用。给模板增加名为 footer 的插槽，并提供两个同名直接子元素。检查：按页面顺序分配，parentNode 仍为宿主；移除两项后显示回退。再把一个提供者包进普通 &lt;div&gt;，解释分配为何改变。

（3）复制 declarative.html 为 practice-declarative.html，移除 &lt;head&gt; 的脚本引用。检查：支持 DSD 的浏览器仍显示 open、closed 的内部说明，状态区保持初始文字；说明为什么 closed 的 shadowRoot 为 null 不能证明没有影子根。

### 提示

第一题在副本片段中查找新增段落；第二题检查宿主的直接子节点；第三题对照源码和 DOM，不把状态区未更新误判为浏览器不支持。练习副本用后可删除。

## 参考与引用来源

- WHATWG HTML：[&lt;template&gt;](https://html.spec.whatwg.org/multipage/scripting.html#the-template-element) 的 content、惰性内容和 shadowrootmode；[&lt;slot&gt;](https://html.spec.whatwg.org/multipage/scripting.html#the-slot-element) 的回退与 assignedNodes/assignedElements 算法；[脚本元素](https://html.spec.whatwg.org/multipage/scripting.html#the-script-element)的 defer；[自定义元素](https://html.spec.whatwg.org/multipage/custom-elements.html#custom-elements)的有效名称与构造器约束。
- WHATWG DOM：[Shadow trees](https://dom.spec.whatwg.org/#shadow-trees)、[Finding slots and slottables](https://dom.spec.whatwg.org/#finding-slots-and-slotables)、[Slottable](https://dom.spec.whatwg.org/#mixin-slotable) 与 [Signaling slot change](https://dom.spec.whatwg.org/#signaling-slot-change)，支持直接子节点、名称匹配、flatten、closed 访问与事件时机。
- MDN：[cloneNode](https://developer.mozilla.org/en-US/docs/Web/API/Node/cloneNode#using_clonenode_with_templates)、[appendChild](https://developer.mozilla.org/en-US/docs/Web/API/Node/appendChild#description)、[importNode](https://developer.mozilla.org/en-US/docs/Web/API/Document/importNode)，支持复制、片段插入、监听器及文档上下文；[Using custom elements](https://developer.mozilla.org/en-US/docs/Web/API/Web_components/Using_custom_elements#implementing_a_custom_element)、[Using shadow DOM](https://developer.mozilla.org/en-US/docs/Web/API/Web_components/Using_shadow_DOM#element.shadowroot_and_the_mode_option)、[slotchange](https://developer.mozilla.org/en-US/docs/Web/API/HTMLSlotElement/slotchange_event)，支持注册、访问边界和后代文字变化；[attachShadow](https://developer.mozilla.org/en-US/docs/Web/API/Element/attachShadow#calling_this_method_on_an_element_that_is_already_a_shadow_host)，支持对已有声明式根的处理。
- CSS Working Group：[CSS Shadow Module Level 1](https://drafts.csswg.org/css-shadow-1/) §3.2、§4.1，支持 :host、::slotted、普通选择器边界和插槽继承。
- Google web.dev：[Declarative Shadow DOM](https://web.dev/articles/declarative-shadow-dom) 的 Browser Support、Parser-only、Feature detection and browser support，以及 Custom Elements and detecting existing Shadow Roots，支持基本版本条件、静态声明与动态属性的区别。
- Python 3.12：[http.server 命令行](https://docs.python.org/3.12/library/http.server.html#command-line-interface)，支持预览端口、绑定地址和服务根目录。